# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is provided via a Croissant schema URL, which contains detailed machine-actionable dataset metadata, data resources, and field definitions.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Published: ", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview

Review available record sets and their `@id`s, then inspect fields and their corresponding IDs. This helps identify how to reference different data resources and select which record sets to load.

In [ ]:
# List all record sets by their @id and name
print("Available RecordSets in dataset:")
for rs in metadata.record_sets:
    print(f"@id: {rs.id}  |  name: {rs.name}")

# Examine the fields for one record set (e.g., the main protein table)
selected_record_set_id = metadata.record_sets[0].id  # select the first RecordSet by default
print(f"\nFields for RecordSet '@id': {selected_record_set_id}")
for field in metadata.record_sets[0].fields:
    print(f"  @id: {field.id:40}  name: {field.name:30} dt: {getattr(field, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from selected record sets into pandas DataFrames for analysis, referencing each by `@id`.

*(Update `record_sets_to_extract` to analyze additional record sets as desired.)*

In [ ]:
# Choose which RecordSets (@ids) to extract
record_sets_to_extract = [rs.id for rs in metadata.record_sets]  # use all
dataframes = {}

for record_set_id in record_sets_to_extract:
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df

# Show column names (field @ids) and preview for first record set
print(f"Columns in first RecordSet (@id={record_sets_to_extract[0]}):\n", dataframes[record_sets_to_extract[0]].columns.tolist())
dataframes[record_sets_to_extract[0]].head()

## 4. Exploratory Data Analysis (EDA)

Apply common EDA and data processing steps to a numeric field. All column names used below are the field `@id`s as found above.

*Below, we analyze the molecular weight field, group by "Sample", and normalize the data. You may adjust which field is filtered or grouped by referencing field `@id`s from above.*

In [ ]:
# Pick RecordSet and numeric field by @id (adjust as needed for your dataset)
record_set_id = record_sets_to_extract[0]

# Find a typical numeric field (e.g., molecular_weight or similar)
candidate_numeric_fields = [c for c in dataframes[record_set_id].columns if 'weight' in c.lower() or 'coverage' in c.lower() or 'count' in c.lower() or 'abundance' in c.lower()]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    # fallback to first float or int field
    sample_row = dataframes[record_set_id].iloc[0]
    for col in dataframes[record_set_id].columns:
        if isinstance(sample_row[col], (float, int, np.integer, np.floating)):
            numeric_field = col
            break
        else:
            numeric_field = None
if not numeric_field:
    raise ValueError('Could not find a numeric field for EDA. Check your fields.')

print(f"Chosen numeric field for EDA: {numeric_field}")

threshold = dataframes[record_set_id][numeric_field].mean() if dataframes[record_set_id][numeric_field].dtype.kind in 'fi' else 0
# Drop missing before filtering
filtered_df = dataframes[record_set_id].dropna(subset=[numeric_field])
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records where '{numeric_field}' > {threshold:.4f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping if a categorical field exists (e.g., Sample or similar)
candidate_group_fields = [c for c in dataframes[record_set_id].columns if 'sample' in c.lower() or 'type' in c.lower() or 'group' in c.lower()]
group_field = candidate_group_fields[0] if candidate_group_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean().reset_index()
    print(f"\nGrouped average of {numeric_field} by {group_field} (first 5 groups):")
    print(grouped_df.head())
else:
    print("No obvious grouping field found for this dataset.")

## 5. Visualization

Visualize the distribution and relationships of key dataset fields.

*Below, we plot the distribution of the main numeric field and demonstrate a boxplot by group if applicable.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field exists, boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook we demonstrated how to:
- Load a FAIR² dataset using its Croissant metadata via the `mlcroissant` library
- Discover available record sets and field `@id`s
- Extract and process tabular records for data analysis
- Use `@id`-based referencing for reproducible, programmatic data access
- Apply EDA and visualize scientific tabular data from mass spectrometry experiments

You may now extend this notebook for deeper analysis or apply FAIR principles to new datasets supported by the Croissant ecosystem.